# 06 · LLM forecasting experiments (Claude)

**Design**
- 16 forecast dates (every 4th test origin, 30 Jun 2025 → 24 Aug 2026), 2 independent runs per prompt, averaged. The proposal planned 3 runs; this was cut to 2 to save time.
- **Experiment 1 – anonymized:** the model sees only the last 156 weekly values, rescaled so the latest value = 100. No name, units or dates. Forecasts are converted back to USD/gal by multiplying by the last price / 100.
- **Experiment 2 – full context:** the model is told the series is California regular gasoline, sees dated prices and a *market brief built only from data available on the forecast date* (Brent, U.S. price, CA–U.S. premium, West Coast stocks and refinery inputs, events to date, season). No news text was used, so the prompt itself cannot leak hindsight.
- Every forecast asks for a 12-week median plus 10th/90th percentiles in JSON.

**How the runs were made:** each forecast was a fresh Claude session (Claude Opus 5.5, `claude-opus-5-5`) started from the Claude app on 14 Oct 2026, told to read exactly one prompt file and use no other tools. All prompts are in `results/llm/prompts/`, all raw replies in `results/llm/responses.jsonl`. The prompts can be rerun through the Anthropic API with any model (see `llm_api_run()` at the end).

**Look-ahead caveat:** Claude Opus 5.5's training data runs to about mid-2026, which overlaps the test period. The full-context experiment can therefore draw on remembered prices/events for origins before ~July 2026; the anonymized experiment is the clean test of pattern-recognition skill.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import *
import plots; plots.setup()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
df = load_weekly()
import json
from llm_prompts import parse, LLM_DIR, PROMPT_DIR, llm_origins
meta = pd.read_csv(LLM_DIR / "prompt_index.csv", parse_dates=["origin_date"]).set_index("prompt_id")
print(len(meta), "prompts;", meta.origin_date.nunique(), "forecast dates")

32 prompts; 16 forecast dates


## Example prompts

In [2]:
print((PROMPT_DIR / "anon_20260309.txt").read_text()[:900], "...\n")
print((PROMPT_DIR / "context_20260309.txt").read_text()[-2600:])

You are a forecasting model. Use only the information in this message. Do not use tools, browse, or look anything up. Reply with JSON only.

Below are 156 consecutive weekly observations of a time series, oldest first. Values are an index scaled so the most recent observation equals 100.

91.42, 90.99, 90.81, 91.93, 92.77, 92.17, 91.52, 91.07, 90.61, 90.50, 91.16, 91.26, 92.46, 92.07, 91.28, 91.32, 91.79, 92.70, 92.46, 94.80, 96.23, 97.80, 100.18, 100.26, 101.04, 103.16, 108.95, 111.90, 115.86, 110.03, 106.01, 102.55, 99.47, 97.07, 95.62, 93.64, 92.66, 90.24, 87.51, 85.59, 85.82, 87.94, 87.55, 85.20, 83.90, 84.31, 85.73, 86.83, 87.71, 87.18, 91.32, 92.15, 92.95, 94.27, 96.29, 100.96, 103.49, 102.83, 101.55, 100.45, 99.08, 97.49, 96.05, 94.40, 91.87, 90.30, 89.61, 89.00, 89.04, 88.10, 86.94, 85.80, 85.51, 85.02, 84.78, 84.92, 86.51, 87.37, 88.65, 88.16, 86.63, 86.55, 86.94, 86.51, 84.94,  ...

3
2025-03-10,4.481
2025-03-17,4.432
2025-03-24,4.418
2025-03-31,4.585
2025-04-07,4.731
2025-04

## Parse logged replies and build forecasts

In [3]:
rows = [json.loads(l) for l in open(LLM_DIR / "responses.jsonl")]
recs, issues = [], []
for r in rows:
    o, why = parse(r["raw"])
    try:
        json.loads(r["raw"])
    except Exception:
        issues.append((r["prompt_id"], r["rep"], "malformed JSON (stray brace) - arrays recovered by parser"))
    if not why:
        issues.append((r["prompt_id"], r["rep"], "no rationale returned"))
    if (o["p10"] > o["median"]).any() or (o["p90"] < o["median"]).any():
        issues.append((r["prompt_id"], r["rep"], "quantiles out of order"))
    recs.append({"prompt_id": r["prompt_id"], "rep": r["rep"], **o, "rationale": why})
issues.append(("context_20260629", 2, "session made 2 tool calls instead of 1 (forecast kept; flagged)"))
print(pd.Series({"replies": len(recs), "prompts": len({x['prompt_id'] for x in recs})}))
pd.DataFrame(issues, columns=["prompt_id", "rep", "note"])

replies    64
prompts    32
dtype: int64


,prompt_id,rep,note
0,anon_20251117,1,no rationale returned
1,context_20250922,2,no rationale returned
2,context_20251215,2,malformed JSON (stray brace) - arrays recovere...
3,context_20260112,2,no rationale returned
4,context_20260406,2,malformed JSON (stray brace) - arrays recovere...
5,context_20260629,2,session made 2 tool calls instead of 1 (foreca...


In [4]:
by = {}
for x in recs:
    by.setdefault(x["prompt_id"], []).append(x)
for exp in ["anon", "context"]:
    out = []
    for pid, lst in sorted(by.items()):
        if not pid.startswith(exp):
            continue
        m = meta.loc[pid]
        s = m.scale / 100 if exp == "anon" else 1.0
        avg = {k: np.mean([x[k] for x in lst], axis=0) * s for k in ("median", "p10", "p90")}
        out.append(make_fc(f"llm_{exp}", m.origin_date, avg["median"], avg["p10"], avg["p90"]))
    pd.concat(out).sort_values(["origin_date", "horizon"]).to_csv(FC_DIR / f"llm_{exp}.csv", index=False, date_format="%Y-%m-%d")
print("wrote forecasts/llm_anon.csv and forecasts/llm_context.csv")

wrote forecasts/llm_anon.csv and forecasts/llm_context.csv


## Run-to-run consistency and what the model said

In [5]:
sd = []
for pid, lst in by.items():
    if len(lst) == 2:
        m = meta.loc[pid]; s = m.scale / 100 if pid.startswith("anon") else 1.0
        sd.append({"experiment": pid.split("_")[0], "abs diff between runs, h=4 ($)": abs(lst[0]["median"][3] - lst[1]["median"][3]) * s,
                   "abs diff, h=12 ($)": abs(lst[0]["median"][11] - lst[1]["median"][11]) * s})
display(pd.DataFrame(sd).groupby("experiment").mean().round(3))
guessed = [x for x in recs if x["prompt_id"].startswith("anon") and any(w in x["rationale"].lower() for w in ["gasoline", "fuel", "petrol"])]
print(f"Anonymized runs whose rationale names fuel/gasoline anyway: {len(guessed)} of {sum(p.startswith('anon') for p in by for _ in by[p])}")
for x in guessed:
    print(" -", x["prompt_id"], ":", x["rationale"][:160])

,"abs diff between runs, h=4 ($)","abs diff, h=12 ($)"
experiment,,
anon,0.027,0.066
context,0.029,0.042


Anonymized runs whose rationale names fuel/gasoline anyway: 2 of 32
 - anon_20250630 : The series has drifted down from a recent local peak and is close to the bottom of its range; this resembles retail gasoline in summer, so I forecast flat to mi
 - anon_20251117 : The series looks like a weekly fuel-price index that usually dips from late fall into the holidays and then firms up again in late winter, so the median drifts 


## Accuracy on the 16 LLM dates (all models scored on the same dates)

In [6]:
fc = attach_actuals(load_forecasts(), df)
dates = meta.origin_date.unique()
same = fc[fc.origin_date.isin(dates) & fc.horizon.isin(SCORED_H)]
t = same.groupby(["horizon", "model"]).apply(metrics)[["n", "MAE", "RMSE", "MAPE", "cover80"]]
t["skill_vs_naive"] = t["MAE"] / t.index.get_level_values("horizon").map(t["MAE"].xs("naive", level="model"))
t = t.round(3).sort_values(["horizon", "MAE"])
t.to_csv(RES_DIR / "llm_same_dates.csv"); t

n    MAE   RMSE    MAPE  cover80  skill_vs_naive
horizon model                                                               
4       arimax           16.0  0.209  0.266   4.038    0.812           0.747
        arima_fourier    16.0  0.221  0.273   4.258    0.750           0.791
        arimax_lags1to4  16.0  0.221  0.271   4.256    0.750           0.789
        llm_context      16.0  0.239  0.299   4.661    0.688           0.852
        ets              16.0  0.261  0.328   5.163    0.625           0.933
        moirai2          16.0  0.279  0.361   5.369    0.688           0.997
        naive            16.0  0.280  0.366   5.518    0.750           1.000
        chronos_bolt     16.0  0.286  0.360   5.575    0.625           1.021
        llm_anon         16.0  0.291  0.372   5.651    0.562           1.041
        seasonal_naive   16.0  0.622  0.836  11.275    1.000           2.220
12      arimax           14.0  0.377  0.546   6.970    0.786           0.613
        arima_fourier    14.0  0.399  0.562   7.385    0.786           0.648
        arimax_lags1to4  14.0  0.404  0.562   7.467    0.786           0.656
        llm_context      14.0  0.471  0.661   8.504    0.714           0.766
        llm_anon         14.0  0.567  0.796  10.380    0.571           0.921
        chronos_bolt     14.0  0.583  0.823  10.723    0.643           0.946
        moirai2          14.0  0.611  0.868  11.200    0.714           0.993
        naive            14.0  0.616  0.801  11.453    0.786           1.000
        ets              14.0  0.667  0.862  12.546    0.643           1.084
        seasonal_naive   14.0  0.699  0.893  12.612    1.000           1.135

In [7]:
gap = t.xs("llm_anon", level="model")["MAE"] - t.xs("llm_context", level="model")["MAE"]
print("anon MAE - context MAE (positive = background knowledge / context helps):"); print(gap.round(3).to_string())
pre = same[same.origin_date < "2026-07-01"]; post = same[same.origin_date >= "2026-07-01"]
for lbl, d in [("origins before Jul 2026 (inside Claude's training window)", pre), ("origins Jul 2026+ (after training data)", post)]:
    print("\n" + lbl)
    print(d[d.model.isin(["llm_anon", "llm_context", "naive", "arimax"])].groupby(["horizon", "model"]).error.apply(lambda e: e.abs().mean()).unstack(0).round(3).to_string())

anon MAE - context MAE (positive = background knowledge / context helps):
horizon
4     0.052
12    0.096

origins before Jul 2026 (inside Claude's training window)
horizon         4      12
model                    
arimax       0.202  0.377
llm_anon     0.281  0.567
llm_context  0.219  0.471
naive        0.278  0.616

origins Jul 2026+ (after training data)
horizon          4
model             
arimax       0.264
llm_anon     0.363
llm_context  0.377
naive        0.296


## Re-running through the API (optional)
The cell below re-runs every prompt through the Anthropic API so anyone can reproduce or extend the experiment with another model (needs `pip install anthropic` and `ANTHROPIC_API_KEY`). It is not executed here.

In [8]:
def llm_api_run(model="claude-opus-5-5", reps=2, out=LLM_DIR / "responses_api.jsonl"):
    import anthropic, datetime
    client = anthropic.Anthropic()
    with open(out, "a") as f:
        for p in sorted(PROMPT_DIR.glob("*.txt")):
            text = p.read_text()
            for rep in range(1, reps + 1):
                msg = client.messages.create(model=model, max_tokens=800, messages=[{"role": "user", "content": text}])
                f.write(json.dumps({"prompt_id": p.stem, "rep": rep, "model": model,
                                    "logged_at": datetime.datetime.now(datetime.UTC).isoformat(),
                                    "raw": msg.content[0].text}) + "\n")
# llm_api_run()   # uncomment to run